In [ ]:
import pandas as pd
import numpy as np
import ast
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import os
import matplotlib.pyplot as plt
import seaborn as sns
from plinder_analysis_utils import DockingAnalysisBase, PoseBustersAnalysis, PropertyAnalysis


## Load Data

In [ ]:
PLINDER_TEST_COLUMNS = [
    "system_id", "ligand_smiles",
    # binary 
    "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    "system_num_pocket_residues", "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions",
]
# Create category mapping for visualization
CATEGORY_MAPPING = {
    "ligand_is_covalent": "binary",
    "ligand_is_ion": "binary",
    "ligand_is_cofactor": "binary",
    "ligand_is_artifact": "binary",
    "system_num_protein_chains": "discrete",
    "ligand_num_rot_bonds": "continuous",    
    "ligand_num_hbd": "continuous",
    "ligand_num_hba": "continuous",
    "ligand_num_rings": "discrete",
    "entry_resolution": "continuous",
    "entry_validation_molprobity": "continuous",
    "system_num_pocket_residues": "continuous",
    "system_num_interactions": "continuous",
    "ligand_molecular_weight": "continuous",
    "ligand_crippen_clogp": "continuous",
    "ligand_num_interacting_residues": "continuous",
    "ligand_num_neighboring_residues": "continuous",
    "ligand_num_interactions": "continuous",
    "ligand_is_artifact": "binary"         
}

In [ ]:
from Approach import DiffDockApproach, DiffDockPocketApproach, GninaApproach, SurfDockApproach, ICMApproach, ChaiApproach
exp_name = "plinder_set_0"
approaches = [
    DiffDockApproach(),
    DiffDockPocketApproach(),
    GninaApproach(),
    SurfDockApproach(),
    ICMApproach(),
    ChaiApproach(),
]
df_all = []
for approach in approaches:
    method_name = approach.get_name()
    print(method_name)
    df_method  = pd.read_csv(f"{method_name}_{exp_name}_results.csv")
    print(df_method.shape)
    df_all.append(df_method)

df_combined = pd.concat(df_all, ignore_index=True)
print(df_combined.shape)

In [ ]:
# get top 10 surfdock poses ranked by rmsd
df_combined[df_combined['method'] == 'surfdock'] \
    .sort_values(by='rmsd', ascending=False) \
    .head(10)[['protein', 'rmsd']]

In [ ]:
inputs_csv = pd.read_csv('./plidner_test_input.csv')
# Convert strings to lists using ast.literal_eval
inputs_csv['ion_paths'] = inputs_csv['ion_paths'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
inputs_csv['has_ion'] = inputs_csv['ion_paths'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
print(inputs_csv.shape)
inputs_csv.head()
test_annot_df = pd.read_csv('plinder_test.csv')
print(test_annot_df['system_id'].nunique())
print(test_annot_df.shape)
test_annot_df = test_annot_df[PLINDER_TEST_COLUMNS]

In [ ]:
test_annot_df.drop_duplicates(subset=['system_id', 'ligand_smiles'], inplace=True)
print(test_annot_df.shape)

# Filter rows where ligand_is_ion is True
ion_df = test_annot_df[test_annot_df['ligand_is_ion']]
# Inspect the result
print(f"Found {len(ion_df)} ion ligands:")
test_annot_df = test_annot_df[~test_annot_df['ligand_is_ion']]
print(test_annot_df['system_id'].shape)
print(test_annot_df['system_id'].nunique())

# test_annot_df = test_annot_df[~test_annot_df['ligand_is_cofactor']]
# print(test_annot_df['system_id'].shape)
# print(test_annot_df['system_id'].nunique())

# test_annot_df = test_annot_df[~test_annot_df['ligand_is_covalent']]
# print(test_annot_df['system_id'].shape)
# print(test_annot_df['system_id'].nunique())

test_annot_df = test_annot_df[~test_annot_df['ligand_is_artifact']]
print(test_annot_df['system_id'].shape)
print(test_annot_df['system_id'].nunique())


In [ ]:
# 0. ensure df_combined carries ligand_smile
df_combined['ligand_smile'] = df_combined['protein'].map(
    inputs_csv.set_index('system_id')['ligand_description']
)

# 1. find system_ids with >1 annotation row
dupe_ids = (
    test_annot_df['system_id']
    .loc[test_annot_df['system_id'].duplicated(keep=False)]
    .unique()
)

# 2. split df_combined
simple = df_combined.loc[~df_combined['protein'].isin(dupe_ids)]
dup    = df_combined.loc[ df_combined['protein'].isin(dupe_ids)]

# 3. merge simple cases on system_id only
merged_simple = simple.merge(
    test_annot_df.drop_duplicates('system_id'),
    left_on='protein',
    right_on='system_id',
    how='left',
    suffixes=('','_annot')
)

# 4. merge dup cases on (system_id,ligand_smile)
merged_dup = dup.merge(
    test_annot_df,
    left_on=['protein','ligand_smile'],
    right_on=['system_id','ligand_smiles'],
    how='left',
    suffixes=('','_annot')
)

# 5. recombine
df_merged = pd.concat([merged_simple, merged_dup], ignore_index=True)

print(df_merged.shape)
df_merged.head()
df_merged[PLINDER_TEST_COLUMNS].isnull().sum()

In [ ]:
# build a lookup of fingerprints for each (system_id, annotation_smiles)
test_annot_df['mol'] = test_annot_df['ligand_smiles'].map(Chem.MolFromSmiles)
test_annot_df['fp']  = test_annot_df['mol'].map(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))

# function to find best‐match SMILES for a given row
def fuzzy_merge_row(row, candidates, threshold=0.85):
    mol = Chem.MolFromSmiles(row['ligand_smile'])
    if mol is None: return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    best = None; best_score = threshold
    for idx, cand in candidates.iterrows():
        sim = DataStructs.TanimotoSimilarity(fp, cand['fp'])
        if sim > best_score:
            best_score, best = sim, idx
    return best

# identify rows in df_merged with null annotation
mask = df_merged['ligand_is_covalent'].isnull()  # pick any _annot column
rows = df_merged.loc[mask, ['protein','ligand_smile']].copy()

# for each, look up test‐annotations for that system_id and pick the best‐fingerprint match
for i, r in rows.iterrows():
    candidates = test_annot_df[test_annot_df['system_id']==r['protein']]
    match_idx = fuzzy_merge_row(r, candidates)
    if match_idx is not None:
        # copy in the missing _annot columns from that test_annot_df row
        for c in PLINDER_TEST_COLUMNS[2:]:
            df_merged.at[i, c+'_annot'] = test_annot_df.at[match_idx, c]

# clean up helper cols
test_annot_df.drop(columns=['mol','fp'], inplace=True)
df_merged[PLINDER_TEST_COLUMNS].isnull().sum()

In [ ]:
# Drop any row where *all* of the PLINDER_TEST_COLUMNS are null
df_merged = df_merged.dropna(
    subset=PLINDER_TEST_COLUMNS,
    how='all'
)

print(df_merged.shape)
df_merged.head()
df_merged[PLINDER_TEST_COLUMNS].isnull().sum()

In [ ]:
df_merged.to_csv('plinder_test_merged.csv', index=False)

In [ ]:
# 0. add ligand_smile from inputs_csv into df_combined
df_combined['ligand_smile'] = df_combined['protein'].map(
    inputs_csv.set_index('system_id')['ligand_description']
)
allowed = (
    df_combined[['protein','ligand_smile']]
    .drop_duplicates()
    .rename(columns={'protein':'system_id','ligand_smile':'ligand_smiles'})
)

df_merged = pd.merge(
    df_combined,
    test_annot_df.drop_duplicates(subset=['system_id']),
    left_on=['protein'],
    right_on=['system_id'],
    how='left',
)

print(df_merged.shape)
df_merged[PLINDER_TEST_COLUMNS].isnull().sum()

In [ ]:
from rdkit import Chem

def canonical_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol, isomericSmiles=True) if mol else None
    except:
        return None

# 0. add ligand_smile from inputs_csv into df_combined
df_combined['ligand_smile'] = df_combined['protein'].map(
    inputs_csv.set_index('system_id')['ligand_description']
)

# 1. Add a canonical‐SMILES column to both dataframes
test_annot_df['canonical_smiles'] = test_annot_df['ligand_smiles'].map(canonical_smiles)
df_combined['canonical_smiles'] = df_combined['ligand_smile'].map(canonical_smiles)

# 2. Deduplicate the annotations by (system_id, canonical_smiles)
test_annot_unique = test_annot_df.drop_duplicates(['system_id', 'canonical_smiles'])

# 3. Pick the annotation columns to merge in, EXCLUDING the join keys
cols_to_merge = [
    c for c in test_annot_unique.columns
    if c not in ('system_id', 'ligand_smiles', 'canonical_smiles')
]

# 4. Merge on protein↔system_id and canonical_smiles
df_combined = df_combined.merge(
    test_annot_unique[['system_id', 'canonical_smiles'] + cols_to_merge],
    left_on=['protein', 'canonical_smiles'],
    right_on=['system_id', 'canonical_smiles'],
    how='left',
    suffixes=('', '_annot')
)

# 5. Clean up helper columns
df_combined.drop(columns=['canonical_smiles'], inplace=True)


In [ ]:
df_combined[cols_to_merge].isnull().sum()

In [ ]:
df_combined[df_combined['ligand_smile'] == 'NC(=O)C1=CN([C@@H]2O[C@H](CO[P@](=O)(O)O[P@@](=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1']

In [ ]:
test_annot_df['canonical_smiles'] = test_annot_df['ligand_smiles'].map(canonical_smiles)
inputs_csv['canonical_smiles'] = inputs_csv['ligand_description'].map(canonical_smiles)

In [ ]:
print(test_annot_df['ligand_smiles'].nunique(), inputs_csv['ligand_description'].nunique())
print(len(set(test_annot_df['ligand_smiles']).intersection(set(inputs_csv['ligand_description']))))
print(list((set(test_annot_df['canonical_smiles']).difference(set(inputs_csv['canonical_smiles']))))[:1])

In [ ]:
# create a boolean mask for SMILES shorter than 10 characters
short_smiles_mask = test_annot_df['ligand_smiles'].str.len() < 7

# count them
short_smiles_count = short_smiles_mask.sum()
print(f"Number of rows with ligand_smiles length < 7: {short_smiles_count}")
# print the rows with short SMILES
print(test_annot_df[short_smiles_mask]['ligand_smiles'])

In [ ]:
# Identify columns with "smile" in their name
smile_cols = [col for col in test_annot_df.columns if 'smile' in col.lower()]
print("Columns containing 'smile':", smile_cols)

# Subset the dataframe to only those columns
test_annot_df[smile_cols].head()
# Find rows where the SMILES columns are not all identical
smile_cols = ['ligand_smiles', 'ligand_resolved_smiles', '']
mask = test_annot_df[smile_cols].nunique(axis=1) > 1
diff_smiles_df = test_annot_df.loc[mask, smile_cols + ['system_id']]
print(f"Found {len(diff_smiles_df)} rows with differing SMILES:")
diff_smiles_df

In [ ]:
# Load the Processed DataFrame
df_combined = pd.read_csv('plinder_test_combined.csv')
"has_ion" in df_combined.columns

## PB Analysis

In [ ]:
# Initialize the analysis object
analysis = PoseBustersAnalysis(df_combined)

# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="diffdock_pocket_only",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

In [ ]:
# List of methods to include in analysis
methods = ['diffdock_pocket_only', 'chai-1', 'icm', 'gnina', 'surfdock',]

# Run the universal analysis
universal_results = analysis.analyze_universal(
    methods=methods,
    rmsd_threshold=2.0,
    plot=True
)

# Access key results
success_proteins = universal_results['all_success_proteins']
failure_proteins = universal_results['all_failure_proteins']
significant_metrics = universal_results['significant_metrics']

In [ ]:
comparative_results = analysis.analyze_comparative(
    method1="diffdock",
    method2="icm",
    rmsd_threshold=2.0,
    plot=True
)

# Access results
method1_only = comparative_results['method1_only_proteins']
method2_only = comparative_results['method2_only_proteins']
significant_metrics = comparative_results['significant_metrics']

In [ ]:
ml_methods = ['diffdock_pocket_only', 'surfdock', 'chai-1']
physics_methods = ['gnina', 'icm']

group_results = analysis.analyze_method_group_comparative(
    group1_methods=ml_methods,
    group2_methods=physics_methods,
    group1_label="ML-based",
    group2_label="Physics-based",
    rmsd_threshold=2.0,
    plot=True
)

## Base Property Analysis

### Individual Property Analysis

In [ ]:
# 1. Initialize
property_analysis = PropertyAnalysis(df_combined)

#### gnina

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "gnina", prop, prop_type,
    )

#### icm

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "icm", prop, prop_type,
    )

#### surfdock

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "surfdock", prop, prop_type,
    )

In [ ]:
# get top 10 surfdock poses ranked by rmsd
df_combined[df_combined['method'] == 'surfdock'] \
    .sort_values(by='rmsd', ascending=False) \
    .head(10)[['protein', 'rmsd']]

#### diffdock_pocket_only

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "diffdock_pocket_only", prop, prop_type,
    )

#### chai-1

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "chai-1", prop, prop_type,
    )

### Property Comparative Analysis

In [ ]:
# 3. Compare how properties affect different methods
for prop in PLINDER_TEST_COLUMNS[2:]:
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_comparative(
        "diffdock_pocket_only", "chai-1", prop, prop_type
    )

### Property Universal Analysis

In [ ]:
property_analysis = PropertyAnalysis(df_combined)

# 4. Look at universal patterns across all methods
for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_universal(
        ["icm", "gnina", "surfdock", "diffdock_pocket_only", "chai-1"], prop, prop_type,
    )

# Analysis After Removing covalent, cofactor, ionic systems

In [ ]:
# add the `has_ion` flag from inputs_csv (mapping on system_id)
df_combined['has_ion'] = df_combined['protein'].map(
    inputs_csv.set_index('system_id')['has_ion']
)

# build a boolean mask: drop any row where covalent, ionic or has_ion is True
mask = ~(
    df_combined['ligand_is_covalent'] |
    df_combined['ligand_is_ion'] |
    df_combined['has_ion'] |
    df_combined['ligand_is_cofactor']
)

# filter and reset index
df_combined = df_combined.loc[mask].reset_index(drop=True)
print("Filtered shape:", df_combined.shape)

## Individual and Universal Analysis

In [ ]:
property_analysis = PropertyAnalysis(df_combined)

In [ ]:
PLINDER_TEST_COLUMNS.remove("ligand_is_artifact")
PLINDER_TEST_COLUMNS.remove("ligand_is_cofactor")
PLINDER_TEST_COLUMNS.remove("ligand_is_ion")
PLINDER_TEST_COLUMNS.remove("ligand_is_covalent")
PLINDER_TEST_COLUMNS

### gnina

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]: 
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "gnina", prop, prop_type,
    )

### icm

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]: 
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "icm", prop, prop_type,
    )

### surfdock

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]: 
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "surfdock", prop, prop_type,
    )

### diffdock_pocket_only

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]: 
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "diffdock_pocket_only", prop, prop_type,
    )

### chai-1

In [ ]:
for prop in PLINDER_TEST_COLUMNS[2:]: 
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_single_method(
        "chai-1", prop, prop_type,
    )

## Universal Analysis

In [ ]:
# First analyze multiple properties
property_analysis = PropertyAnalysis(df_combined)
methods = ["surfdock", "gnina", "chai-1", "diffdock_pocket_only", "icm"]

all_results = []

for prop in PLINDER_TEST_COLUMNS[2:]:
    # 2. Explore impact of properties on a single method
    prop_type = CATEGORY_MAPPING[prop]
    prop_analysis = property_analysis.analyze_property_universal(
        methods, prop, prop_type,
    )
    all_results.append(prop_analysis)



### Plotting universal

In [ ]:
# Plot significant differences 
property_analysis.plot_significant_property_differences(
    all_results, 
    p_threshold=0.05,
    sort_by='effect_size'
)